# Assignment 08: Full Data Preprocessing Pipeline (100 points)

## Context

In practice, 80% of an AI project is **data preprocessing**. Raw data is messy: missing values, outliers, inconsistent types, mixed scales. Before any model sees the data, you must clean, validate, transform, and split it -- all while avoiding data leakage.

This assignment walks through a complete preprocessing pipeline on a synthetic messy dataset. Each step builds on the previous one, mirroring the real workflow.

### Pipeline Order

1. **Explore** -- understand what you have
2. **Clean** -- fix invalid values, handle outliers, fill missing data
3. **Split** -- separate train/test BEFORE feature engineering
4. **Engineer** -- normalize, encode, combine features
5. **Verify** -- confirm everything is correct

### Why This Order Matters

Splitting before engineering prevents **data leakage**: test-set information leaking into training statistics. If you compute the mean for normalization on the full dataset, your model has implicitly "seen" test data.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)

> **WARNING !!!**
>
> - Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else** for the following purposes:
>     - **As a part of your final solution.**
>     - **Temporarily import something to assist you to get a solution.**
>
>     **Rule of thumb:** Each part has its particular purpose to intentionally test you something. Do not attempt to find a shortcut to circumvent the rule.
>
> - **No sklearn** -- do everything from scratch with NumPy/Pandas.
> - **No explicit loops** where vectorized alternatives exist.
> - Use `plt.tight_layout()` before `plt.show()`.

---

## Setup: The Messy Dataset

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
N = 200

df_raw = pd.DataFrame({
    'age': np.where(np.random.rand(N) > 0.9, np.nan,
                    np.random.randint(18, 70, N).astype(float)),
    'income': np.where(np.random.rand(N) > 0.85, np.nan,
                       np.random.lognormal(10.5, 0.8, N)),
    'education_years': np.where(np.random.rand(N) > 0.95, np.nan,
                                np.random.choice([12, 14, 16, 18, 20], N).astype(float)),
    'hours_per_week': np.where(np.random.rand(N) > 0.92, np.nan,
                               np.random.normal(40, 12, N).clip(5, 80)),
    'occupation': np.random.choice(
        ['Engineer', 'Teacher', 'Doctor', 'Artist', 'Manager', None],
        N, p=[0.25, 0.2, 0.15, 0.15, 0.2, 0.05]),
    'region': np.random.choice(['North', 'South', 'East', 'West'], N),
    'satisfaction': np.random.randint(1, 11, N),  # 1-10 scale, this is the target
})

# Inject specific problems
df_raw.loc[5, 'income'] = 5000000     # extreme outlier
df_raw.loc[10, 'age'] = -5            # invalid negative age
df_raw.loc[15, 'hours_per_week'] = 200  # impossible value

print("Raw dataset shape:", df_raw.shape)
print("\nFirst 5 rows:")
print(df_raw.head())
print("\nMissing values:")
print(df_raw.isna().sum())
print("\nData types:")
print(df_raw.dtypes)

---

## Part 1 (15 points, coding task)

**Reasoning is not required.**

Perform exploratory data analysis (EDA) to understand the data before cleaning.

1. **(3 pts)** Print summary statistics for all numeric columns using `.describe()`.

2. **(4 pts)** Create a figure with a `2x2` grid of histograms for the four numeric columns (`age`, `income`, `education_years`, `hours_per_week`). Figure size: `(10, 8)`. Each subplot should have a title matching the column name.

3. **(4 pts)** Print value counts for the categorical columns (`occupation`, `region`). Count and print the number of `None`/NaN values in `occupation`.

4. **(4 pts)** Write your findings as code comments: identify the outliers/invalid values, which columns have missing data, and any data type issues.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """

---

Now that you understand the data, clean it systematically. The order matters: fix invalid values first (so they become NaN), then handle outliers, then fill remaining missing values. If you fill before fixing invalids, you contaminate your fill statistics.

---

## Part 2 (20 points, coding task)

**Reasoning is not required.**

Clean the dataset step by step. Work on a copy of `df_raw`.

1. **(4 pts)** Replace invalid values with NaN: `age < 0` or `age > 120`, and `hours_per_week > 100`.

2. **(4 pts)** Handle outliers in `income`: cap values at the 99th percentile (winsorization). Use `np.nanpercentile` to compute the threshold, then clip.

3. **(4 pts)** Fill missing numeric values (`age`, `income`, `education_years`, `hours_per_week`) with the column median.

4. **(4 pts)** Fill missing categorical values (`occupation`) with the string `'Unknown'`.

5. **(4 pts)** Drop any remaining rows with NaN. Print the final shape and confirm zero missing values.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

df = df_raw.copy()

""" END OF THIS PART """

---

The next step is splitting. This must happen BEFORE any feature engineering that computes statistics (means, stds, category frequencies). Computing statistics on the full dataset and then splitting is data leakage -- a subtle but critical error.

---

## Part 3 (15 points, coding task)

**Reasoning is not required.**

Split the cleaned data into training and test sets.

1. **(5 pts)** Create shuffled indices using `np.random.permutation`. Compute the split point at 80% of the data.

2. **(5 pts)** Split `df` into `df_train` and `df_test` DataFrames. Print their shapes.

3. **(5 pts)** Verify the split: confirm that `df_train` and `df_test` together have the same number of rows as `df`, and that there is no overlap in indices.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

df_train = None
df_test = None

""" END OF THIS PART """

---

Now we can safely compute statistics from the training set and apply transformations to both sets. This is where the fit/transform pattern from Assignment 07 pays off.

---

## Part 4 (30 points, coding task)

**Reasoning is not required.**

Engineer features from the split data. Fit all statistics on the training set only.

In [ ]:
"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""
target_col = 'satisfaction'
numeric_cols = ['age', 'income', 'education_years', 'hours_per_week']
categorical_cols = ['occupation', 'region']

1. **(8 pts)** Z-score normalize the numeric features. Compute mean and std from `df_train` only, then transform both `df_train` and `df_test`. Store the normalized numeric arrays.

2. **(8 pts)** One-hot encode the categorical features (`occupation`, `region`). Determine the unique categories from `df_train`. Ensure both train and test have the **same columns** (test may have unknown categories -- assign them all-zero vectors).

3. **(7 pts)** Separate the target variable `'satisfaction'`. Concatenate the normalized numeric features and one-hot encoded categorical features into final NumPy arrays:
   - `X_train` and `X_test` (features)
   - `y_train` and `y_test` (targets)

4. **(7 pts)** Print the final shapes. Verify: no NaN values, numeric columns have mean $\approx 0$ and std $\approx 1$ in training data, one-hot columns are binary.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

X_train = None  # NumPy array
y_train = None  # NumPy array
X_test = None   # NumPy array
y_test = None   # NumPy array

""" END OF THIS PART """

---

The final step is verification and visualization. Always check your work: confirm shapes, check for NaN, verify normalization, and visualize distributions. A preprocessing bug that goes unnoticed will silently degrade every downstream model.

---

## Part 5 (20 points, coding task)

**Reasoning is not required.**

Verify your preprocessing and create summary visualizations.

1. **(4 pts)** Assert that there are no NaN values in `X_train`, `X_test`, `y_train`, `y_test`. Print confirmation.

2. **(4 pts)** Print the mean and std of each numeric feature column in `X_train` (the first `len(numeric_cols)` columns). Confirm mean $\approx 0$ and std $\approx 1$.

3. **(4 pts)** Verify that the one-hot encoded columns (columns after the numeric ones) contain only 0s and 1s. For each original categorical variable, verify that its one-hot columns sum to exactly 1 per row in the training data.

4. **(8 pts)** Create a 2-panel figure (`(12, 5)`):
   - **Left panel**: Overlapping histograms of the target variable for train and test sets, with legend
   - **Right panel**: Correlation heatmap of the numeric features (training set only), annotated with values

In [ ]:
### WRITE YOUR SOLUTION HERE ###

""" END OF THIS PART """